# 07_02 — Features Históricas Sem Leakage

Este notebook cria features históricas para o modelo **early-stage** sem vazar informação do futuro.

Entrada:
- `data/processed/abt_early_stage_v1_dev.parquet`
- `data/processed/abt_early_stage_v1_holdout.parquet`
- `data/processed/feature_list_early_stage_v1.txt`

Saída:
- `data/processed/abt_early_stage_v1_dev_hist.parquet`
- `data/processed/abt_early_stage_v1_holdout_hist.parquet`
- `data/processed/abt_early_stage_v1_full_hist.parquet`
- `data/processed/feature_list_early_stage_v1_hist.txt`

Regra de leakage:
- No `dev`, cada linha usa apenas processos com `Datainicial` anterior.
- No `holdout`, as estatísticas são fitadas no `dev` e aplicadas no `holdout`.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

pd.set_option("display.max_columns", 400)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 260)

BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "outputs" / "reports" / "modelagem_early_stage"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DEV_FILE = PROCESSED_DIR / "abt_early_stage_v1_dev.parquet"
HOLDOUT_FILE = PROCESSED_DIR / "abt_early_stage_v1_holdout.parquet"
FEATURE_LIST_FILE = PROCESSED_DIR / "feature_list_early_stage_v1.txt"

TARGET_COL = "target_banco_ganhou"
DATE_COL = "Datainicial"
VALUE_COL = "fe_valor_ajuizado"

SMOOTHING_M = 30
RARE_THRESHOLD = 30

print("Setup concluído.")

## 2. Carregar bases

In [ ]:
if not DEV_FILE.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {DEV_FILE}")

if not HOLDOUT_FILE.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {HOLDOUT_FILE}")

df_dev = pd.read_parquet(DEV_FILE)
df_holdout = pd.read_parquet(HOLDOUT_FILE)

df_dev[DATE_COL] = pd.to_datetime(df_dev[DATE_COL], errors="coerce")
df_holdout[DATE_COL] = pd.to_datetime(df_holdout[DATE_COL], errors="coerce")

df_dev[TARGET_COL] = df_dev[TARGET_COL].astype(int)
df_holdout[TARGET_COL] = df_holdout[TARGET_COL].astype(int)

df_dev[VALUE_COL] = pd.to_numeric(df_dev[VALUE_COL], errors="coerce")
df_holdout[VALUE_COL] = pd.to_numeric(df_holdout[VALUE_COL], errors="coerce")

if FEATURE_LIST_FILE.exists():
    with open(FEATURE_LIST_FILE, "r", encoding="utf-8") as f:
        feature_list_v1 = [line.strip() for line in f if line.strip()]
else:
    feature_list_v1 = [
        c for c in df_dev.columns
        if c not in ["Numerodistribuicao", "Identificador", DATE_COL, TARGET_COL]
    ]

print("Dev:", df_dev.shape)
print("Holdout:", df_holdout.shape)
print("Features etapa 1:", len(feature_list_v1))
df_dev.head()

## 3. Funções auxiliares

In [ ]:
def save_report(df_report, filename):
    path = REPORTS_DIR / filename
    df_report.to_csv(path, index=False)
    print(f"Relatório salvo em: {path}")

def clean_colname(name):
    name = str(name)
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r"[^a-zA-Z0-9_]", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_").lower()

def existing_cols(df, cols):
    return [c for c in cols if c in df.columns]

def safe_divide(a, b):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    return np.where((b == 0) | pd.isna(b), np.nan, a / b)

def add_date_only(df, date_col=DATE_COL):
    out = df.copy()
    out["_date_only_hist"] = pd.to_datetime(out[date_col], errors="coerce").dt.floor("D")
    return out

def global_prior_by_date(df, date_col=DATE_COL, target_col=TARGET_COL, value_col=VALUE_COL):
    temp = add_date_only(df, date_col)

    daily = (
        temp
        .groupby("_date_only_hist", dropna=False)
        .agg(
            global_daily_count=(target_col, "size"),
            global_daily_gain_sum=(target_col, "sum"),
            global_daily_value_sum=(value_col, "sum")
        )
        .reset_index()
        .sort_values("_date_only_hist")
    )

    daily["global_prior_count"] = daily["global_daily_count"].cumsum().shift(1).fillna(0)
    daily["global_prior_gain_sum"] = daily["global_daily_gain_sum"].cumsum().shift(1).fillna(0)
    daily["global_prior_value_sum"] = daily["global_daily_value_sum"].cumsum().shift(1).fillna(0)
    daily["global_prior_loss_sum"] = daily["global_prior_count"] - daily["global_prior_gain_sum"]

    overall_gain_rate = temp[target_col].mean()
    overall_loss_rate = 1 - overall_gain_rate
    overall_value_mean = temp[value_col].mean()

    daily["global_prior_gain_rate"] = np.where(
        daily["global_prior_count"] > 0,
        daily["global_prior_gain_sum"] / daily["global_prior_count"],
        overall_gain_rate
    )

    daily["global_prior_loss_rate"] = np.where(
        daily["global_prior_count"] > 0,
        daily["global_prior_loss_sum"] / daily["global_prior_count"],
        overall_loss_rate
    )

    daily["global_prior_value_mean"] = np.where(
        daily["global_prior_count"] > 0,
        daily["global_prior_value_sum"] / daily["global_prior_count"],
        overall_value_mean
    )

    return daily[[
        "_date_only_hist",
        "global_prior_count",
        "global_prior_gain_rate",
        "global_prior_loss_rate",
        "global_prior_value_mean",
    ]], overall_gain_rate, overall_loss_rate, overall_value_mean

## 4. Selecionar grupos históricos

In [ ]:
candidate_group_cols = [
    "fe_nm_assunto_norm",
    "fe_nm_produto_norm",
    "fe_nm_acao_norm",
    "fe_uf_norm",
    "fe_comarca_norm",
    "fe_carteira_norm",
    "fe_nm_empresa_norm",
    "fe_classe_texto_norm",
    "fe_area_texto_norm",
    "fe_assunto_normalizado_texto_norm",
    "fe_assunto_texto_norm",
    "fe_cidade_norm",
    "fe_orgao_julgador_texto_norm",
    "fe_tipo_de_processo_texto_norm",
    "fe_tipo_de_justica_texto_norm",
    "fe_inter_nm_produto__nm_acao",
    "fe_inter_nm_produto__nm_assunto",
    "fe_inter_nm_acao__nm_assunto",
    "fe_inter_nm_produto__nm_acao__nm_assunto",
    "fe_inter_uf__comarca",
    "fe_inter_uf__nm_assunto",
    "fe_inter_comarca__nm_assunto",
    "fe_inter_carteira__nm_assunto",
    "fe_inter_classe_texto__assunto_normalizado_texto",
    "fe_inter_tipo_de_justica_texto__assunto_normalizado_texto",
]

group_cols = existing_cols(df_dev, candidate_group_cols)

group_report = pd.DataFrame({
    "group_col": group_cols,
    "nunique_dev": [df_dev[c].nunique(dropna=True) for c in group_cols],
    "missing_rate_dev": [df_dev[c].isna().mean() for c in group_cols],
})

save_report(group_report, "10_historical_group_cols_selected.csv")

print("Qtd grupos históricos selecionados:", len(group_cols))
group_report

## 5. Função de features históricas strict-time para o Dev

In [ ]:
def create_strict_time_historical_features_dev(
    df,
    group_col,
    date_col=DATE_COL,
    target_col=TARGET_COL,
    value_col=VALUE_COL,
    m=SMOOTHING_M,
    rare_threshold=RARE_THRESHOLD,
):
    # Cria features históricas no dev usando apenas datas anteriores.
    if group_col not in df.columns:
        raise KeyError(f"Coluna {group_col} não existe.")

    base = add_date_only(df, date_col)
    prefix = clean_colname(group_col)

    global_prior, overall_gain_rate, overall_loss_rate, overall_value_mean = global_prior_by_date(
        base,
        date_col=date_col,
        target_col=target_col,
        value_col=value_col
    )

    daily_group = (
        base
        .groupby([group_col, "_date_only_hist"], dropna=False)
        .agg(
            daily_count=(target_col, "size"),
            daily_gain_sum=(target_col, "sum"),
            daily_value_sum=(value_col, "sum"),
        )
        .reset_index()
        .sort_values([group_col, "_date_only_hist"])
    )

    daily_group[f"{prefix}_hist_count"] = (
        daily_group.groupby(group_col, dropna=False)["daily_count"].cumsum()
        - daily_group["daily_count"]
    )

    daily_group[f"{prefix}_hist_gain_sum"] = (
        daily_group.groupby(group_col, dropna=False)["daily_gain_sum"].cumsum()
        - daily_group["daily_gain_sum"]
    )

    daily_group[f"{prefix}_hist_value_sum"] = (
        daily_group.groupby(group_col, dropna=False)["daily_value_sum"].cumsum()
        - daily_group["daily_value_sum"]
    )

    daily_group = daily_group.merge(global_prior, on="_date_only_hist", how="left")

    hist_count = daily_group[f"{prefix}_hist_count"]
    hist_gain_sum = daily_group[f"{prefix}_hist_gain_sum"]
    hist_loss_sum = hist_count - hist_gain_sum
    hist_value_sum = daily_group[f"{prefix}_hist_value_sum"]

    prior_global_gain = daily_group["global_prior_gain_rate"].fillna(overall_gain_rate)
    prior_global_loss = daily_group["global_prior_loss_rate"].fillna(overall_loss_rate)
    prior_global_value = daily_group["global_prior_value_mean"].fillna(overall_value_mean)

    daily_group[f"{prefix}_hist_gain_rate_smooth"] = (
        hist_gain_sum + m * prior_global_gain
    ) / (hist_count + m)

    daily_group[f"{prefix}_hist_loss_rate_smooth"] = (
        hist_loss_sum + m * prior_global_loss
    ) / (hist_count + m)

    daily_group[f"{prefix}_hist_value_mean"] = np.where(
        hist_count > 0,
        hist_value_sum / hist_count,
        prior_global_value
    )

    daily_group[f"{prefix}_hist_log_count"] = np.log1p(hist_count)
    daily_group[f"{prefix}_hist_is_rare"] = (hist_count < rare_threshold).astype(int)

    feature_cols = [
        f"{prefix}_hist_count",
        f"{prefix}_hist_log_count",
        f"{prefix}_hist_is_rare",
        f"{prefix}_hist_gain_rate_smooth",
        f"{prefix}_hist_loss_rate_smooth",
        f"{prefix}_hist_value_mean",
    ]

    merge_cols = [group_col, "_date_only_hist"] + feature_cols

    out = base[[group_col, "_date_only_hist", value_col]].merge(
        daily_group[merge_cols],
        on=[group_col, "_date_only_hist"],
        how="left"
    )

    out[f"{prefix}_ratio_value_vs_hist_mean"] = safe_divide(
        out[value_col],
        out[f"{prefix}_hist_value_mean"]
    )

    out[f"{prefix}_flag_value_above_hist_mean"] = (
        out[value_col] > out[f"{prefix}_hist_value_mean"]
    ).astype(int)

    feature_cols += [
        f"{prefix}_ratio_value_vs_hist_mean",
        f"{prefix}_flag_value_above_hist_mean",
    ]

    return out[feature_cols], feature_cols

## 6. Aplicar features históricas no Dev

In [ ]:
df_dev_hist = df_dev.copy()

historical_feature_cols = []
historical_dev_diagnostics = []

for group_col in group_cols:
    print(f"Criando histórico strict-time para: {group_col}")

    feats, feat_cols = create_strict_time_historical_features_dev(
        df_dev_hist,
        group_col=group_col,
        date_col=DATE_COL,
        target_col=TARGET_COL,
        value_col=VALUE_COL,
        m=SMOOTHING_M,
        rare_threshold=RARE_THRESHOLD,
    )

    for c in feat_cols:
        df_dev_hist[c] = feats[c].values

    historical_feature_cols.extend(feat_cols)

    historical_dev_diagnostics.append({
        "group_col": group_col,
        "qtd_features_criadas": len(feat_cols),
        "missing_rate_medio_features": feats.isna().mean().mean(),
        "nunique_group_dev": df_dev_hist[group_col].nunique(dropna=True),
    })

historical_dev_diagnostics = pd.DataFrame(historical_dev_diagnostics)
save_report(historical_dev_diagnostics, "11_historical_features_dev_diagnostics.csv")

print("Qtd features históricas criadas no dev:", len(historical_feature_cols))
historical_dev_diagnostics

## 7. Funções fit no Dev / apply no Holdout

In [ ]:
def fit_historical_maps_on_dev(
    df_train,
    group_col,
    target_col=TARGET_COL,
    value_col=VALUE_COL,
    m=SMOOTHING_M,
    rare_threshold=RARE_THRESHOLD,
):
    prefix = clean_colname(group_col)

    global_gain_rate = df_train[target_col].mean()
    global_loss_rate = 1 - global_gain_rate
    global_value_mean = df_train[value_col].mean()

    stats = (
        df_train
        .groupby(group_col, dropna=False)
        .agg(
            hist_count=(target_col, "size"),
            hist_gain_sum=(target_col, "sum"),
            hist_value_sum=(value_col, "sum")
        )
        .reset_index()
    )

    stats["hist_loss_sum"] = stats["hist_count"] - stats["hist_gain_sum"]

    stats[f"{prefix}_hist_gain_rate_smooth"] = (
        stats["hist_gain_sum"] + m * global_gain_rate
    ) / (stats["hist_count"] + m)

    stats[f"{prefix}_hist_loss_rate_smooth"] = (
        stats["hist_loss_sum"] + m * global_loss_rate
    ) / (stats["hist_count"] + m)

    stats[f"{prefix}_hist_value_mean"] = np.where(
        stats["hist_count"] > 0,
        stats["hist_value_sum"] / stats["hist_count"],
        global_value_mean
    )

    stats[f"{prefix}_hist_count"] = stats["hist_count"]
    stats[f"{prefix}_hist_log_count"] = np.log1p(stats["hist_count"])
    stats[f"{prefix}_hist_is_rare"] = (stats["hist_count"] < rare_threshold).astype(int)

    feature_cols = [
        f"{prefix}_hist_count",
        f"{prefix}_hist_log_count",
        f"{prefix}_hist_is_rare",
        f"{prefix}_hist_gain_rate_smooth",
        f"{prefix}_hist_loss_rate_smooth",
        f"{prefix}_hist_value_mean",
    ]

    mapping = stats[[group_col] + feature_cols].copy()

    defaults = {
        f"{prefix}_hist_count": 0,
        f"{prefix}_hist_log_count": 0,
        f"{prefix}_hist_is_rare": 1,
        f"{prefix}_hist_gain_rate_smooth": global_gain_rate,
        f"{prefix}_hist_loss_rate_smooth": global_loss_rate,
        f"{prefix}_hist_value_mean": global_value_mean,
    }

    return mapping, feature_cols, defaults

def apply_historical_maps(
    df_apply,
    group_col,
    mapping,
    feature_cols,
    defaults,
    value_col=VALUE_COL,
):
    prefix = clean_colname(group_col)

    out = df_apply[[group_col, value_col]].merge(
        mapping,
        on=group_col,
        how="left"
    )

    for c in feature_cols:
        out[c] = out[c].fillna(defaults[c])

    out[f"{prefix}_ratio_value_vs_hist_mean"] = safe_divide(
        out[value_col],
        out[f"{prefix}_hist_value_mean"]
    )

    out[f"{prefix}_flag_value_above_hist_mean"] = (
        out[value_col] > out[f"{prefix}_hist_value_mean"]
    ).astype(int)

    final_cols = feature_cols + [
        f"{prefix}_ratio_value_vs_hist_mean",
        f"{prefix}_flag_value_above_hist_mean",
    ]

    return out[final_cols], final_cols

## 8. Aplicar mapas históricos no Holdout

In [ ]:
df_holdout_hist = df_holdout.copy()

historical_feature_cols_holdout = []
historical_holdout_diagnostics = []

for group_col in group_cols:
    print(f"Fit dev / apply holdout para: {group_col}")

    mapping, base_feat_cols, defaults = fit_historical_maps_on_dev(
        df_dev_hist,
        group_col=group_col,
        target_col=TARGET_COL,
        value_col=VALUE_COL,
        m=SMOOTHING_M,
        rare_threshold=RARE_THRESHOLD,
    )

    feats_holdout, feat_cols_holdout = apply_historical_maps(
        df_holdout_hist,
        group_col=group_col,
        mapping=mapping,
        feature_cols=base_feat_cols,
        defaults=defaults,
        value_col=VALUE_COL,
    )

    for c in feat_cols_holdout:
        df_holdout_hist[c] = feats_holdout[c].values

    historical_feature_cols_holdout.extend(feat_cols_holdout)

    unknown_rate = (~df_holdout_hist[group_col].isin(mapping[group_col])).mean()

    historical_holdout_diagnostics.append({
        "group_col": group_col,
        "qtd_features_criadas": len(feat_cols_holdout),
        "missing_rate_medio_features": feats_holdout.isna().mean().mean(),
        "nunique_group_holdout": df_holdout_hist[group_col].nunique(dropna=True),
        "unknown_rate_holdout_vs_dev": unknown_rate,
    })

historical_holdout_diagnostics = pd.DataFrame(historical_holdout_diagnostics)
save_report(historical_holdout_diagnostics, "12_historical_features_holdout_diagnostics.csv")

print("Qtd features históricas criadas no holdout:", len(historical_feature_cols_holdout))
historical_holdout_diagnostics

## 9. Consistência Dev/Holdout e sanidade

In [ ]:
historical_feature_cols = list(dict.fromkeys(historical_feature_cols))
historical_feature_cols_holdout = list(dict.fromkeys(historical_feature_cols_holdout))

missing_in_holdout = [c for c in historical_feature_cols if c not in df_holdout_hist.columns]
missing_in_dev = [c for c in historical_feature_cols_holdout if c not in df_dev_hist.columns]

consistency_report = pd.DataFrame([{
    "qtd_hist_features_dev": len(historical_feature_cols),
    "qtd_hist_features_holdout": len(historical_feature_cols_holdout),
    "missing_in_holdout": str(missing_in_holdout),
    "missing_in_dev": str(missing_in_dev),
    "same_feature_set": len(missing_in_holdout) == 0 and len(missing_in_dev) == 0,
}])

save_report(consistency_report, "13_historical_feature_consistency_report.csv")
consistency_report

In [ ]:
def feature_sanity_report(df, features, dataset_name):
    rows = []

    for col in features:
        s = df[col]

        if pd.api.types.is_numeric_dtype(s):
            s_num = pd.to_numeric(s, errors="coerce")
            inf_rate = np.isinf(s_num).mean()
            min_val = s_num.replace([np.inf, -np.inf], np.nan).min()
            max_val = s_num.replace([np.inf, -np.inf], np.nan).max()
            mean_val = s_num.replace([np.inf, -np.inf], np.nan).mean()
        else:
            inf_rate = np.nan
            min_val = np.nan
            max_val = np.nan
            mean_val = np.nan

        rows.append({
            "dataset": dataset_name,
            "feature": col,
            "dtype": str(s.dtype),
            "missing_rate": s.isna().mean(),
            "nunique": s.nunique(dropna=True),
            "inf_rate": inf_rate,
            "min": min_val,
            "mean": mean_val,
            "max": max_val,
        })

    return pd.DataFrame(rows)

sanity_dev = feature_sanity_report(df_dev_hist, historical_feature_cols, "dev")
sanity_holdout = feature_sanity_report(df_holdout_hist, historical_feature_cols, "holdout")

sanity_report = pd.concat([sanity_dev, sanity_holdout], ignore_index=True)
save_report(sanity_report, "14_historical_features_sanity_report.csv")

sanity_report.sort_values(["missing_rate", "nunique"], ascending=[False, True]).head(50)

## 10. Features de resumo histórico

In [ ]:
def add_historical_summary_features(df, hist_features):
    out = df.copy()

    loss_rate_cols = [c for c in hist_features if c.endswith("_hist_loss_rate_smooth")]
    gain_rate_cols = [c for c in hist_features if c.endswith("_hist_gain_rate_smooth")]
    count_cols = [c for c in hist_features if c.endswith("_hist_count")]
    rare_cols = [c for c in hist_features if c.endswith("_hist_is_rare")]
    ratio_cols = [c for c in hist_features if c.endswith("_ratio_value_vs_hist_mean")]

    if loss_rate_cols:
        out["fe_hist_loss_rate_mean_across_groups"] = out[loss_rate_cols].mean(axis=1)
        out["fe_hist_loss_rate_max_across_groups"] = out[loss_rate_cols].max(axis=1)
        out["fe_hist_loss_rate_min_across_groups"] = out[loss_rate_cols].min(axis=1)

    if gain_rate_cols:
        out["fe_hist_gain_rate_mean_across_groups"] = out[gain_rate_cols].mean(axis=1)
        out["fe_hist_gain_rate_max_across_groups"] = out[gain_rate_cols].max(axis=1)
        out["fe_hist_gain_rate_min_across_groups"] = out[gain_rate_cols].min(axis=1)

    if count_cols:
        out["fe_hist_count_mean_across_groups"] = out[count_cols].mean(axis=1)
        out["fe_hist_count_max_across_groups"] = out[count_cols].max(axis=1)
        out["fe_hist_count_min_across_groups"] = out[count_cols].min(axis=1)

    if rare_cols:
        out["fe_hist_rare_count_across_groups"] = out[rare_cols].sum(axis=1)
        out["fe_hist_rare_share_across_groups"] = out[rare_cols].mean(axis=1)

    if ratio_cols:
        ratio_df = out[ratio_cols].replace([np.inf, -np.inf], np.nan)
        out["fe_hist_ratio_value_mean_across_groups"] = ratio_df.mean(axis=1)
        out["fe_hist_ratio_value_max_across_groups"] = ratio_df.max(axis=1)

    summary_features = [c for c in out.columns if c.startswith("fe_hist_")]
    return out, summary_features

df_dev_hist, historical_summary_features = add_historical_summary_features(
    df_dev_hist,
    historical_feature_cols
)

df_holdout_hist, historical_summary_features_holdout = add_historical_summary_features(
    df_holdout_hist,
    historical_feature_cols
)

historical_summary_features = list(dict.fromkeys(historical_summary_features))

print("Features de resumo histórico:", len(historical_summary_features))
historical_summary_features

## 11. Atualizar lista de features e checar leakage por nome

In [ ]:
historical_feature_cols = list(dict.fromkeys(historical_feature_cols))

all_features_early_v1_hist = []

for col in feature_list_v1 + historical_feature_cols + historical_summary_features:
    if col in df_dev_hist.columns and col not in all_features_early_v1_hist:
        all_features_early_v1_hist.append(col)

feature_dict_hist = pd.DataFrame({
    "feature": all_features_early_v1_hist,
    "tipo": [
        "historical_group" if f in historical_feature_cols else
        "historical_summary" if f in historical_summary_features else
        "rowwise_step_1"
        for f in all_features_early_v1_hist
    ],
    "dtype_dev": [str(df_dev_hist[f].dtype) for f in all_features_early_v1_hist],
    "missing_rate_dev": [df_dev_hist[f].isna().mean() for f in all_features_early_v1_hist],
    "missing_rate_holdout": [df_holdout_hist[f].isna().mean() for f in all_features_early_v1_hist],
    "nunique_dev": [df_dev_hist[f].nunique(dropna=True) for f in all_features_early_v1_hist],
    "nunique_holdout": [df_holdout_hist[f].nunique(dropna=True) for f in all_features_early_v1_hist],
})

save_report(feature_dict_hist, "15_feature_dictionary_early_v1_hist.csv")

print("Qtd features etapa 1:", len(feature_list_v1))
print("Qtd features históricas por grupo:", len(historical_feature_cols))
print("Qtd features resumo histórico:", len(historical_summary_features))
print("Qtd features total:", len(all_features_early_v1_hist))

feature_dict_hist.head(100)

In [ ]:
leakage_patterns = [
    r"^target_",
    r"^desfecho",
    r"^valor_perdido",
    r"^valor_ganho",
    r"resultado",
    r"sentenca",
    r"condenacao",
    r"acordo_prob",
    r"procedente_prob",
    r"procedente_parc",
    r"dano_moral_prob",
    r"provavel_",
    r"texto_sentenca",
    r"texto_tutela",
    r"transito",
    r"data_condenacao",
    r"data_sentenca",
    r"data_execucao",
    r"valor_do_acordo",
    r"valor_arbitrado",
]

blocked_exact = {
    "Fasedoprocesso",
    "fase_processual_texto",
    "status_texto",
    "status_tipo",
    "Qtddias_Ultimoandamento",
    "Passiveldeacordo",
    "Processorelevante",
    "Estimativa_De_Perda",
    "Credenciado",
    "Adv_Interno",
    "Advogadoadverso",
}

def is_name_leakage(feature):
    c = str(feature).lower()

    if feature in blocked_exact:
        return True

    for pat in leakage_patterns:
        if re.search(pat, c):
            return True

    return False

leakage_check_hist = pd.DataFrame({
    "feature": all_features_early_v1_hist,
    "possivel_leakage_por_nome": [is_name_leakage(f) for f in all_features_early_v1_hist],
})

save_report(leakage_check_hist, "16_feature_leakage_check_early_v1_hist.csv")

display(leakage_check_hist["possivel_leakage_por_nome"].value_counts())
leakage_check_hist[leakage_check_hist["possivel_leakage_por_nome"] == True]

## 12. Salvar bases com features históricas

In [ ]:
cols_id = existing_cols(df_dev_hist, [
    "Numerodistribuicao",
    "Identificador",
    DATE_COL,
    TARGET_COL,
])

cols_to_save = list(dict.fromkeys(cols_id + all_features_early_v1_hist))

df_dev_hist_out = df_dev_hist[cols_to_save].copy()
df_holdout_hist_out = df_holdout_hist[cols_to_save].copy()

dev_hist_path = PROCESSED_DIR / "abt_early_stage_v1_dev_hist.parquet"
holdout_hist_path = PROCESSED_DIR / "abt_early_stage_v1_holdout_hist.parquet"
full_hist_path = PROCESSED_DIR / "abt_early_stage_v1_full_hist.parquet"
feature_list_hist_path = PROCESSED_DIR / "feature_list_early_stage_v1_hist.txt"

df_dev_hist_out.to_parquet(dev_hist_path, index=False)
df_holdout_hist_out.to_parquet(holdout_hist_path, index=False)

pd.concat([df_dev_hist_out, df_holdout_hist_out], axis=0).to_parquet(
    full_hist_path,
    index=False
)

with open(feature_list_hist_path, "w", encoding="utf-8") as f:
    for feature in all_features_early_v1_hist:
        f.write(feature + "\n")

print("Dev hist salvo em:", dev_hist_path, df_dev_hist_out.shape)
print("Holdout hist salvo em:", holdout_hist_path, df_holdout_hist_out.shape)
print("Full hist salvo em:", full_hist_path)
print("Feature list hist salvo em:", feature_list_hist_path)

## 13. Resumo final da Etapa 2

In [ ]:
summary_step_2 = pd.DataFrame([{
    "qtd_linhas_dev": len(df_dev_hist_out),
    "qtd_linhas_holdout": len(df_holdout_hist_out),
    "qtd_features_step_1": len(feature_list_v1),
    "qtd_historical_group_features": len(historical_feature_cols),
    "qtd_historical_summary_features": len(historical_summary_features),
    "qtd_features_total_hist": len(all_features_early_v1_hist),
    "target_rate_dev": df_dev_hist_out[TARGET_COL].mean(),
    "target_rate_holdout": df_holdout_hist_out[TARGET_COL].mean(),
    "dev_hist_path": str(dev_hist_path),
    "holdout_hist_path": str(holdout_hist_path),
    "feature_list_hist_path": str(feature_list_hist_path),
}])

save_report(summary_step_2, "17_summary_step_2_historical_features.csv")
summary_step_2.T

# O que verificar após rodar

Traga para a próxima iteração:

1. `summary_step_2.T`
2. Quantidade de features históricas criadas.
3. `historical_holdout_diagnostics`, principalmente `unknown_rate_holdout_vs_dev`.
4. Se alguma feature apareceu como `possivel_leakage_por_nome = True`.
5. As 20 primeiras linhas de `feature_dict_hist`.
6. Se houve erro de memória ou lentidão.

Se esta etapa estiver correta, a próxima será:

## Etapa 3 — Walk-forward temporal

Nela vamos validar o modelo respeitando o tempo, recalculando features históricas dentro de cada fold.